In [1]:
using LinearAlgebra
using ProgressMeter
using JLD2
using Mosek
using Ket

using ppt2

include("utils.jl")

mprint (generic function with 1 method)

In [4]:
# Load precomputed forms
files_4x4 = [
    "../pncp_forms_4x4.jld2",
    "../pncp_forms_4x4_batch2.jld2",
    "../pncp_forms_4x4_batch3.jld2",
    "../pncp_forms_4x4_batch4.jld2",
]

files_3x3 = [
    "../pncp_forms_3x3.jld2",
]

forms_4x4 = vcat([jldopen(file) do file
    vcat([file[k] for k in keys(file)]...)
end for file in files_4x4]...)

forms_3x3 = vcat([jldopen(file) do file
    vcat([file[k] for k in keys(file)]...)
end for file in files_3x3]...)

forms = Dict((4, 4) => forms_4x4, (3, 3) => forms_3x3)

Dict{Tuple{Int64, Int64}, Vector{Matrix{Float64}}} with 2 entries:
  (3, 3) => [[25.9968 1.71845 … 12.219 -10.238; 1.71845 18.4385 … -4.06508 8.54…
  (4, 4) => [[1.62892 1.14988 … 1.50244 3.93037; 1.14988 80.294 … 7.93861 20.94…

In [5]:
function ispos(form, n::Int, m::Int; attempts=100000, tol=1e-6)
    s = Inf
    for _ in 1:attempts
        x = randn(n)
        y = randn(m)
        xy = kron(x, y)

        val = xy' * form * xy
        s = min(s, val)
        if s < -tol
            return s, x, y
        end
    end
    return s, nothing, nothing
end

function istrpos(form, n::Int, m::Int; attempts=100000, tol=1e-6)
    s = Inf
    for _ in 1:attempts
        xy = rand_sep(n, m)

        val = tr(form * xy)
        s = min(s, val)
        if s < -tol
            return s, xy
        end
    end
    return s, nothing
end

istrpos (generic function with 1 method)

In [ ]:
function test_ppt2_pncp(n::Int, m::Int, forms; n_trials::Int=10000, rank::Int=0, tol::Float64=1e-6)
    @showprogress for i in 1:n_trials
        psd = rand_psd(n, m; r=rank)
        cpp = ampliation(psd, psd, n, m)
        r, idx = findmin(tr.(forms .* Ref(cpp)))
        if r < -tol
            println("Witness $idx detects entanglement: $r")
            return psd, forms[idx]
        end
    end
    return nothing, nothing
end

function test_ppt2_dps(n::Int, m::Int; n_trials::Int=10000, rank::Int=0, tol::Float64=1e-6)
    @showprogress for i in 1:n_trials
        psd = rand_psd(n, m; r=rank)
        cpp = Hermitian(ampliation(psd, psd, n, m))
        r, w = entanglement_robustness(cpp, [n, m], 2; solver=Mosek.Optimizer)
        if r > tol
            println("DPS detects entanglement: $r")
            return psd, w
        end
    end
    return nothing, nothing
end

function diagnostics(psd, wit, n::Int, m::Int)
    println("Eigvals of channel:")
    println(eigvals(psd))
    cpp = Hermitian(ampliation(psd, psd, n, m))
    println("Eigvals of composite:")
    println(eigvals(cpp))
    println("Eigvals of witness:")
    println(eigvals(wit))
    println("Positivity of witness map:")
    println(ispos(wit, n, m, attempts=10000000, tol=1e-6))
    println("Positivity on separable states:")
    println(istrpos(wit, n, m, attempts=1000000, tol=1e-6))
    println("Expectation value of the witness on the CPP:")
    println(tr(wit * cpp))
end

diagnostics (generic function with 1 method)

In [20]:
psd4, wit4 = test_ppt2_dps(4, 4; n_trials=1000, rank=6)

DPS detects entanglement: 27.989185284093143


([5.513172325875708 0.8759222691900768 … 0.30053551784806964 1.9276419208857367; 0.8759222691900768 1.0806038604148953 … -1.0985660801145314 0.24781958216629646; … ; 0.30053551784806964 -1.0985660801145314 … 1.8292917799632087 0.41306813320417557; 1.9276419208857367 0.24781958216629646 … 0.41306813320417557 1.684168246225925], [0.0037481137835697806 0.001447353553063306 … 0.09039252953059643 -0.07916121643791256; 0.001447353553063306 0.0005589030769610761 … 0.32133257695565864 -0.2814068573692133; … ; 0.09039252953059643 0.32133257695565864 … 0.00015865948518953066 0.0017842114554497588; -0.07916121643791256 -0.2814068573692133 … 0.0017842114554497588 0.020064425469709074])

In [21]:
diagnostics(psd4, wit4, 4, 4)

Eigenvalues of the state:
[-7.514035807285183e-15, -2.5741625816606535e-15, -1.4524174675995364e-15, -6.860192383412134e-16, -4.2494003328146267e-16, -9.969204216424278e-17, 1.610512782655196e-16, 2.421257670415707e-16, 5.424197779843205e-16, 2.5660439751753924e-15, 5.639814526905075, 7.170740372269647, 9.328660579236676, 16.415370659259935, 20.51136948494631, 38.868683773859686]
Eigenvalues of the CPP:
[16.362140886064232, 20.56323307234122, 30.390079494301855, 44.81098928625213, 54.75166714240241, 63.95802152032838, 93.06762928203506, 101.62165729613847, 131.80007206432964, 146.6238890355786, 156.4349395463386, 196.0367073527596, 236.46827835793877, 277.34070649746945, 346.9817268960201, 475.8924810945003]
Eigenvalues of the witness:
[-0.4896301377792155, -0.07101995733309376, -0.061256880450967735, -0.025291628030754056, -0.021814801012659132, -0.0031641970347896587, 0.001126833918405472, 0.003164197064383856, 0.008885198659851906, 0.021814801059413295, 0.025291628135831724, 0.06125

In [22]:
psd3, wit3 = test_ppt2_dps(3, 3; n_trials=1000, rank=9)

Progress: 100%|█████████████████████████████████████████| Time: 0:00:00


DPS detects entanglement: 11.719656367375856


([7.911290864520021 -3.051967691098844 … 1.3863937164520719 0.14954456633848456; -3.051967691098844 6.514859919838614 … -7.281084752172314 2.297634185467386; … ; 1.3863937164520719 -7.281084752172314 … 16.13085059874307 -6.412023560051587; 0.14954456633848456 2.297634185467386 … -6.412023560051587 6.357766384011099], [0.3056830155641711 0.23383469263249562 … -0.060079739612368725 -0.0396639440499975; 0.23383469263249562 0.1788737374343049 … 0.07279851108231056 0.04806072858577784; … ; -0.060079739612368725 0.07279851108231056 … 0.02962773175706074 -0.07023196187166794; -0.0396639440499975 0.04806072858577784 … -0.07023196187166794 0.1664834987913705])

In [23]:
diagnostics(psd3, wit3, 3, 3)

Eigenvalues of the state:
[0.00500360364315085, 0.1987915196003904, 0.9039263235863937, 1.5105832083073674, 3.67074128570379, 7.029687150755063, 13.432850449583466, 27.102749795890738, 35.88664645391455]
Eigenvalues of the CPP:
[36.30086587149077, 70.16099311199142, 89.26765002315342, 143.50104552093907, 214.12780943283124, 347.5496318320332, 400.9762739781027, 410.97741544004776, 763.2823294896997]
Eigenvalues of the witness:
[-0.3923658640473608, -0.00342794159789583, -0.0016607111856295823, 1.4508275641201935e-5, 0.0016607115290467247, 0.0034279455985151645, 0.19008667493693454, 0.39236586550986, 0.8098988104808009]
Is the witness positive?
(-2.9903948561907586e-8, nothing, nothing)
Is the witness separable?
(1.1760783388391899e-7, nothing)
Expectation value of the witness on the CPP:
-11.71965644707424


In [ ]:
psd4_pncp, wit4_pncp = test_ppt2_pncp(4, 4, forms_4x4; n_trials=10000, rank=4)

Progress: 100%|█████████████████████████████████████████| Time: 0:00:04


Witness 210 detects entanglement: -656.7728846465448


([13.40563438504421 0.6929909923887756 … -5.284502504537934 1.84295425384701; 0.6929909923887756 0.6197066908148952 … -0.8380009097860829 -0.058494692554301876; … ; -5.284502504537934 -0.8380009097860829 … 3.0791321893633294 0.04861779077082096; 1.84295425384701 -0.058494692554301876 … 0.04861779077082096 1.7061606902764348], [38.55605859098826 -0.3968073940061877 … -20.78243721350735 20.912822237384653; -0.3968073940061877 11.210553945442761 … -1.5231686679389238 -5.270171756845074; … ; -20.78243721350735 -1.5231686679389238 … 42.55308052186769 -31.56404564441018; 20.912822237384653 -5.270171756845074 … -31.56404564441018 28.158196928200475])

In [ ]:
diagnostics(psd4_pncp, wit4_pncp, 4, 4)

Eigenvalues of the state:
[-7.514035807285183e-15, -2.5741625816606535e-15, -1.4524174675995364e-15, -6.860192383412134e-16, -4.2494003328146267e-16, -9.969204216424278e-17, 1.610512782655196e-16, 2.421257670415707e-16, 5.424197779843205e-16, 2.5660439751753924e-15, 5.639814526905075, 7.170740372269647, 9.328660579236676, 16.415370659259935, 20.51136948494631, 38.868683773859686]
Eigenvalues of the CPP:
[16.362140886064232, 20.56323307234122, 30.390079494301855, 44.81098928625213, 54.75166714240241, 63.95802152032838, 93.06762928203506, 101.62165729613847, 131.80007206432964, 146.6238890355786, 156.4349395463386, 196.0367073527596, 236.46827835793877, 277.34070649746945, 346.9817268960201, 475.8924810945003]
Eigenvalues of the witness:
[-0.4896301377792155, -0.07101995733309376, -0.061256880450967735, -0.025291628030754056, -0.021814801012659132, -0.0031641970347896587, 0.001126833918405472, 0.003164197064383856, 0.008885198659851906, 0.021814801059413295, 0.025291628135831724, 0.06125

In [ ]:
psd3_pncp, wit3_pncp = test_ppt2_pncp(3, 3, forms_3x3; n_trials=1000, rank=4)

Progress: 100%|█████████████████████████████████████████| Time: 0:00:00


Witness 5120 detects entanglement: -472.0519687464793


([1.33833522488488 0.018931343288976055 … -0.4911063681041077 2.5548005264255305; 0.018931343288976055 5.223519438433061 … 1.5678386464663274 -5.357653675876284; … ; -0.4911063681041077 1.5678386464663274 … 0.8205808621412194 -2.335406594672122; 2.5548005264255305 -5.357653675876284 … -2.335406594672122 12.278983131355075], [7.188743911909689 0.02065782874063553 … -0.3199382051828019 -6.12764158404123; 0.02065782874063553 4.35600611069331 … 1.684978625959618 5.241725645988308; … ; -0.3199382051828019 1.684978625959618 … 7.602922949371378 8.518646851031862; -6.12764158404123 5.241725645988308 … 8.518646851031862 12.213815580733147])

In [ ]:
diagnostics(psd3_pncp, wit3_pncp, 3, 3)

Eigenvalues of the state:
[0.00500360364315085, 0.1987915196003904, 0.9039263235863937, 1.5105832083073674, 3.67074128570379, 7.029687150755063, 13.432850449583466, 27.102749795890738, 35.88664645391455]
Eigenvalues of the CPP:
[36.30086587149077, 70.16099311199142, 89.26765002315342, 143.50104552093907, 214.12780943283124, 347.5496318320332, 400.9762739781027, 410.97741544004776, 763.2823294896997]
Eigenvalues of the witness:
[-0.3923658640473608, -0.00342794159789583, -0.0016607111856295823, 1.4508275641201935e-5, 0.0016607115290467247, 0.0034279455985151645, 0.19008667493693454, 0.39236586550986, 0.8098988104808009]
Is the witness positive?
(-2.190410767805208e-8, nothing, nothing)
Is the witness separable?
(9.81804932079916e-7, nothing)
Expectation value of the witness on the CPP:
-11.71965644707424
